
# Sentinel-2 OpenSR-SRGAN vs HAT — Fair x4 Benchmark

## الهدف

تجربة حديثة وقابلة لإعادة الإنتاج لـSentinel-SRGAN، ثم مقارنتها مع HAT على **نفس المهمة ونفس الحزم ونفس بيانات الاختبار**.

## المهمة

```text
Sentinel-2 RGB-NIR at 10 m
           ↓ ×4
RGB-NIR at approximately 2.5 m
```

الحزم:

```text
B02 Blue
B03 Green
B04 Red
B08 NIR
```

## النموذجان

### 1. OpenSR-SRGAN

نستخدم Framework الرسمي من ESA OpenSR، مع الـRGB-NIR preset المنشور. يتضمن:

```text
Generator warm-up
Adversarial ramp-up
EMA
Remote-sensing normalization
Multispectral support
```

### 2. HAT-S2

نحوّل أفضل Backbone من HAT المدربة على WorldView-3 إلى:

```text
4-band Sentinel-2 single-image SR
No PAN input
Scale ×4
```

ثم ندربها على SEN2NAIP.

## التقييم العادل

لن نقارن Sentinel-SRGAN بنتائج HAT الحالية الخاصة بالـPansharpening؛ لأن المهمتين مختلفتان.

المقارنة النهائية تكون على:

```text
OpenSR NAIP
OpenSR SPOT
OpenSR Spain Crops
OpenSR Spain Urban
```

وباستخدام:

```text
PSNR
SSIM
SAM
OpenSR reflectance/spectral/spatial/correctness metrics
```


In [1]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("فعّل T4 GPU.")

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:

from pathlib import Path
import shutil
import subprocess
import importlib.util
import sys

%cd /content

HAT_REPO = Path("/content/HAT")
SRGAN_REPO = Path("/content/OpenSR_SRGAN")

for repo in [HAT_REPO, SRGAN_REPO]:
    if repo.exists():
        shutil.rmtree(repo)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/XPixelGroup/HAT.git",
        str(HAT_REPO),
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/ESAOpenSR/SRGAN.git",
        str(SRGAN_REPO),
    ],
    check=True,
)

!pip install -q opensr-srgan opensr-test huggingface_hub rasterio pytorch-msssim pandas matplotlib tqdm timm einops

original_arch = HAT_REPO / "hat" / "archs" / "hat_arch.py"
standalone_arch = Path("/content/hat_arch_standalone.py")

source = original_arch.read_text(encoding="utf-8")

source = source.replace(
    "from basicsr.utils.registry import ARCH_REGISTRY",
    """class _SimpleRegistry:
    def register(self):
        def decorator(obj):
            return obj
        return decorator
ARCH_REGISTRY = _SimpleRegistry()"""
)

source = source.replace(
    "from basicsr.archs.arch_util import to_2tuple, trunc_normal_",
    "from timm.layers import to_2tuple, trunc_normal_"
)

standalone_arch.write_text(source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "hat_arch_standalone",
    standalone_arch,
)

hat_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hat_module)

HAT = hat_module.HAT

print("HAT and OpenSR-SRGAN installed.")


/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
HAT and OpenSR-SRGAN installed.


In [3]:

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

SEN2NAIP_DIR = (
    PROJECT_DIR
    / "Public_Datasets"
    / "SEN2NAIP"
)

HAT_OUTPUT_DIR = (
    PROJECT_DIR
    / "Sentinel2_HAT_SISR"
)

BENCHMARK_DIR = (
    PROJECT_DIR
    / "Sentinel2_HAT_vs_OpenSR_SRGAN"
)

for folder in [
    SEN2NAIP_DIR,
    HAT_OUTPUT_DIR,
    BENCHMARK_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

HAT_BEST_PATH = (
    HAT_OUTPUT_DIR
    / "best_sentinel2_hat_x4.pth"
)

HAT_LAST_PATH = (
    HAT_OUTPUT_DIR
    / "last_sentinel2_hat_x4.pth"
)

HAT_HISTORY_PATH = (
    HAT_OUTPUT_DIR
    / "training_history.json"
)

BENCHMARK_CSV = (
    BENCHMARK_DIR
    / "opensr_srgan_vs_hat_metrics.csv"
)

BENCHMARK_JSON = (
    BENCHMARK_DIR
    / "opensr_srgan_vs_hat_metrics.json"
)

WV3_SOURCE_CANDIDATES = [
    (
        PROJECT_DIR
        / "WV3_HAT_Long_Pretraining"
        / "best_psnr_wv3_hat_long.pth"
    ),
    (
        PROJECT_DIR
        / "WV3_HAT_Pretraining"
        / "best_wv3_hat_pan_pretrained.pth"
    ),
]

WV3_SOURCE_PATH = next(
    (
        path
        for path in WV3_SOURCE_CANDIDATES
        if path.exists()
    ),
    None,
)

if WV3_SOURCE_PATH is None:
    raise FileNotFoundError(
        "لم يتم العثور على WV3 HAT checkpoint."
    )

print("WV3 source:", WV3_SOURCE_PATH)
print("SEN2NAIP:", SEN2NAIP_DIR)
print("Output:", BENCHMARK_DIR)


Mounted at /content/drive
WV3 source: /content/drive/MyDrive/Super_Resolution_28-07-2026/WV3_HAT_Long_Pretraining/best_psnr_wv3_hat_long.pth
SEN2NAIP: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/SEN2NAIP
Output: /content/drive/MyDrive/Super_Resolution_28-07-2026/Sentinel2_HAT_vs_OpenSR_SRGAN


## تنزيل SEN2NAIP

In [4]:

from huggingface_hub import hf_hub_download
import zipfile
import os

DATA_MODE = "cross_sensor"
# "cross_sensor": 2,851 real Sentinel-2/NAIP pairs.
# "synthetic": configurable synthetic shards for larger training.

NUM_SYNTHETIC_SHARDS = 2
# Increase gradually up to 18 after checking Drive capacity.

repo_id = "isp-uv-es/SEN2NAIP"

if DATA_MODE == "cross_sensor":
    requested_files = [
        "cross-sensor/cross-sensor.zip"
    ]

elif DATA_MODE == "synthetic":
    requested_files = [
        f"synthetic/synthetic_{index:02d}.zip"
        for index in range(
            1,
            NUM_SYNTHETIC_SHARDS + 1,
        )
    ]

else:
    raise ValueError(DATA_MODE)

for remote_name in requested_files:
    local_zip = Path(
        hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=remote_name,
            local_dir=str(SEN2NAIP_DIR),
        )
    )

    extraction_marker = local_zip.with_suffix(
        local_zip.suffix + ".extracted"
    )

    if not extraction_marker.exists():
        print("Extracting:", local_zip)

        with zipfile.ZipFile(local_zip, "r") as archive:
            archive.extractall(
                SEN2NAIP_DIR / "extracted"
            )

        extraction_marker.write_text(
            "done",
            encoding="utf-8",
        )

print("Dataset download/extraction completed.")


Dataset download/extraction completed.


## اكتشاف أزواج LR/HR تلقائيًا

In [5]:

import hashlib
import rasterio

EXTRACTED_DIR = SEN2NAIP_DIR / "extracted"

all_tifs = sorted(
    list(EXTRACTED_DIR.rglob("*.tif"))
    + list(EXTRACTED_DIR.rglob("*.tiff"))
)

def find_pair_files(files):
    by_parent = {}

    for path in files:
        by_parent.setdefault(
            path.parent,
            [],
        ).append(path)

    pairs = []

    for parent, parent_files in by_parent.items():
        lower_names = {
            path.name.lower(): path
            for path in parent_files
        }

        hr_candidates = [
            path
            for path in parent_files
            if (
                path.stem.lower() == "hr"
                or "high" in path.stem.lower()
                or path.stem.lower().startswith("hr_")
            )
        ]

        lr_candidates = [
            path
            for path in parent_files
            if (
                path.stem.lower() == "lr"
                or "low" in path.stem.lower()
                or path.stem.lower().startswith("lr_")
            )
        ]

        if hr_candidates and lr_candidates:
            pairs.append(
                (
                    lr_candidates[0],
                    hr_candidates[0],
                )
            )

    return sorted(
        pairs,
        key=lambda pair: str(pair[0]),
    )


pairs = find_pair_files(all_tifs)

if not pairs:
    print("Example TIFF files:")

    for path in all_tifs[:30]:
        print(path)

    raise RuntimeError(
        "لم أجد أزواج lr.tif/hr.tif. أرسل قائمة الملفات المطبوعة."
    )

print("Total pairs:", len(pairs))
print("First LR:", pairs[0][0])
print("First HR:", pairs[0][1])

with rasterio.open(pairs[0][0]) as lr_source:
    print(
        "LR:",
        lr_source.count,
        lr_source.height,
        lr_source.width,
        lr_source.dtypes,
    )

with rasterio.open(pairs[0][1]) as hr_source:
    print(
        "HR:",
        hr_source.count,
        hr_source.height,
        hr_source.width,
        hr_source.dtypes,
    )


Total pairs: 2851
First LR: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/SEN2NAIP/extracted/cross-sensor/ROI_0000/lr.tif
First HR: /content/drive/MyDrive/Super_Resolution_28-07-2026/Public_Datasets/SEN2NAIP/extracted/cross-sensor/ROI_0000/hr.tif


LR: 4 121 121 ('int32', 'int32', 'int32', 'int32')
HR: 4 484 484 ('uint8', 'uint8', 'uint8', 'uint8')


## Spatially stable split

In [6]:

def pair_split(pair):
    identifier = str(pair[0].parent).encode("utf-8")
    value = int(
        hashlib.sha1(identifier).hexdigest()[:8],
        16,
    ) % 100

    if value < 85:
        return "train"
    if value < 95:
        return "val"
    return "test"


train_pairs = [
    pair for pair in pairs
    if pair_split(pair) == "train"
]

val_pairs = [
    pair for pair in pairs
    if pair_split(pair) == "val"
]

test_pairs = [
    pair for pair in pairs
    if pair_split(pair) == "test"
]

print(
    "Train/Val/Test:",
    len(train_pairs),
    len(val_pairs),
    len(test_pairs),
)

if min(
    len(train_pairs),
    len(val_pairs),
    len(test_pairs),
) == 0:
    raise RuntimeError(
        "Split produced an empty subset."
    )


Train/Val/Test: 2432 302 117


## Dataset loader

In [7]:

import json
import math
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

RUN_PROFILE = "smoke"
# "smoke", "long", "max"

SEED = 42
SCALE = 4
LR_CROP_SIZE = 64

BATCH_SIZE = 1
ACCUMULATION_STEPS = 4
NUM_WORKERS = 0

if RUN_PROFILE == "smoke":
    TARGET_EPOCHS = 2
elif RUN_PROFILE == "long":
    TARGET_EPOCHS = 80
elif RUN_PROFILE == "max":
    TARGET_EPOCHS = 160
else:
    raise ValueError(RUN_PROFILE)

LEARNING_RATE = 2e-5
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-6
EMA_DECAY = 0.999
GRADIENT_CLIP = 1.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


def normalize_lr(array):
    array = array.astype(np.float32)

    if array.max() > 2:
        array = array / 10000.0

    return np.clip(array, 0, 1)


def normalize_hr(array):
    array = array.astype(np.float32)

    if array.max() > 2:
        if array.max() <= 255:
            array = array / 255.0
        elif array.max() <= 65535:
            array = array / 10000.0

    return np.clip(array, 0, 1)


class SEN2NAIPDataset(Dataset):
    def __init__(self, pair_list, training=False):
        self.pairs = pair_list
        self.training = training

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr_path, hr_path = self.pairs[index]

        with rasterio.open(lr_path) as source:
            lr = source.read(
                indexes=list(
                    range(
                        1,
                        min(source.count, 4) + 1,
                    )
                )
            )

        with rasterio.open(hr_path) as source:
            hr = source.read(
                indexes=list(
                    range(
                        1,
                        min(source.count, 4) + 1,
                    )
                )
            )

        if lr.shape[0] < 4 or hr.shape[0] < 4:
            raise RuntimeError(
                f"Need 4 RGB-NIR bands: {lr_path}"
            )

        lr = torch.from_numpy(
            normalize_lr(lr[:4])
        ).float()

        hr = torch.from_numpy(
            normalize_hr(hr[:4])
        ).float()

        expected_hr_height = lr.shape[-2] * SCALE
        expected_hr_width = lr.shape[-1] * SCALE

        usable_hr_height = min(
            hr.shape[-2],
            expected_hr_height,
        )

        usable_hr_width = min(
            hr.shape[-1],
            expected_hr_width,
        )

        usable_lr_height = usable_hr_height // SCALE
        usable_lr_width = usable_hr_width // SCALE

        lr = lr[
            :,
            :usable_lr_height,
            :usable_lr_width,
        ]

        hr = hr[
            :,
            :usable_lr_height * SCALE,
            :usable_lr_width * SCALE,
        ]

        crop_lr_h = min(
            LR_CROP_SIZE,
            lr.shape[-2],
        )

        crop_lr_w = min(
            LR_CROP_SIZE,
            lr.shape[-1],
        )

        if self.training:
            y = random.randint(
                0,
                lr.shape[-2] - crop_lr_h,
            )

            x = random.randint(
                0,
                lr.shape[-1] - crop_lr_w,
            )
        else:
            y = (
                lr.shape[-2] - crop_lr_h
            ) // 2

            x = (
                lr.shape[-1] - crop_lr_w
            ) // 2

        lr = lr[
            :,
            y:y + crop_lr_h,
            x:x + crop_lr_w,
        ]

        hr = hr[
            :,
            y * SCALE:(y + crop_lr_h) * SCALE,
            x * SCALE:(x + crop_lr_w) * SCALE,
        ]

        if self.training:
            if random.random() < 0.5:
                lr = torch.flip(lr, dims=[2])
                hr = torch.flip(hr, dims=[2])

            if random.random() < 0.5:
                lr = torch.flip(lr, dims=[1])
                hr = torch.flip(hr, dims=[1])

            rotations = random.randint(0, 3)

            if rotations:
                lr = torch.rot90(lr, rotations, dims=[1, 2])
                hr = torch.rot90(hr, rotations, dims=[1, 2])

        return {
            "lr": lr.contiguous(),
            "hr": hr.contiguous(),
        }


train_dataset = SEN2NAIPDataset(train_pairs, training=True)
val_dataset = SEN2NAIPDataset(val_pairs, training=False)
test_dataset = SEN2NAIPDataset(test_pairs, training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

sample = train_dataset[0]

print("LR:", sample["lr"].shape)
print("HR:", sample["hr"].shape)
print("Profile:", RUN_PROFILE)


LR: torch.Size([4, 64, 64])
HR: torch.Size([4, 256, 256])
Profile: smoke


## HAT-S2 architecture and WV3 transfer

In [8]:

class SISRRefinementBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, tensor):
        return tensor + self.block(tensor)


class Sentinel2HAT(nn.Module):
    def __init__(self):
        super().__init__()

        self.hat = HAT(
            upscale=4,
            in_chans=4,
            img_size=64,
            window_size=16,
            compress_ratio=3,
            squeeze_factor=30,
            conv_scale=0.01,
            overlap_ratio=0.5,
            img_range=1.0,
            depths=[6, 6, 6, 6, 6, 6],
            embed_dim=180,
            num_heads=[6, 6, 6, 6, 6, 6],
            mlp_ratio=2,
            upsampler="pixelshuffle",
            resi_connection="1conv",
            use_checkpoint=False,
            drop_path_rate=0.0,
        )

        self.hat.conv_last = nn.Conv2d(
            64,
            4,
            3,
            1,
            1,
        )

        self.refine_head = nn.Conv2d(
            8,
            64,
            3,
            1,
            1,
        )

        self.refine_body = nn.Sequential(
            SISRRefinementBlock(64),
            SISRRefinementBlock(64),
            SISRRefinementBlock(64),
            SISRRefinementBlock(64),
        )

        self.refine_tail = nn.Conv2d(
            64,
            4,
            3,
            1,
            1,
        )

    def forward(self, lr):
        bicubic = F.interpolate(
            lr,
            scale_factor=4,
            mode="bicubic",
            align_corners=False,
        )

        residual = self.hat(lr)
        coarse = bicubic + residual

        refinement = self.refine_tail(
            self.refine_body(
                self.refine_head(
                    torch.cat(
                        [
                            coarse,
                            bicubic,
                        ],
                        dim=1,
                    )
                )
            )
        )

        return (coarse + refinement).clamp(0, 1)


hat_s2 = Sentinel2HAT().to(device)

wv3_checkpoint = torch.load(
    WV3_SOURCE_PATH,
    map_location="cpu",
    weights_only=False,
)

wv3_state = wv3_checkpoint.get(
    "ema_state_dict",
    wv3_checkpoint["model_state_dict"],
)

target_state = hat_s2.state_dict()

skip_keys = {
    "hat.conv_first.weight",
    "hat.conv_first.bias",
    "hat.conv_last.weight",
    "hat.conv_last.bias",
    "refine_head.weight",
    "refine_head.bias",
    "refine_tail.weight",
    "refine_tail.bias",
}

exact_transfers = 0

for key, value in wv3_state.items():
    if (
        key in target_state
        and key not in skip_keys
        and target_state[key].shape == value.shape
    ):
        target_state[key] = value.clone()
        exact_transfers += 1

hat_s2.load_state_dict(target_state, strict=True)


def average_input_kernels(weight, target_channels):
    return weight.mean(
        dim=1,
        keepdim=True,
    ).repeat(
        1,
        target_channels,
        1,
        1,
    ) * (
        weight.shape[1] / target_channels
    )


def average_output_kernels(weight, target_channels):
    return weight.mean(
        dim=0,
        keepdim=True,
    ).repeat(
        target_channels,
        1,
        1,
        1,
    )


with torch.no_grad():
    hat_s2.hat.conv_first.weight.copy_(
        average_input_kernels(
            wv3_state["hat.conv_first.weight"],
            4,
        )
    )

    hat_s2.hat.conv_first.bias.copy_(
        wv3_state["hat.conv_first.bias"]
    )

    hat_s2.hat.conv_last.weight.copy_(
        average_output_kernels(
            wv3_state["hat.conv_last.weight"],
            4,
        )
    )

    hat_s2.hat.conv_last.bias.copy_(
        wv3_state["hat.conv_last.bias"].mean().repeat(4)
    )

    source_refine = wv3_state["refine_head.weight"]

    # Use the WV3 coarse and bicubic groups; omit PAN.
    source_coarse = source_refine[:, 0:8]
    source_bicubic = source_refine[:, 8:16]

    target_refine = torch.cat(
        [
            average_input_kernels(source_coarse, 4),
            average_input_kernels(source_bicubic, 4),
        ],
        dim=1,
    )

    hat_s2.refine_head.weight.copy_(target_refine)
    hat_s2.refine_head.bias.copy_(
        wv3_state["refine_head.bias"]
    )

    hat_s2.refine_tail.weight.copy_(
        average_output_kernels(
            wv3_state["refine_tail.weight"],
            4,
        )
    )

    hat_s2.refine_tail.bias.copy_(
        wv3_state["refine_tail.bias"].mean().repeat(4)
    )

print("Exact transferred tensors:", exact_transfers)
print(
    "HAT-S2 parameters:",
    f"{sum(p.numel() for p in hat_s2.parameters()):,}",
)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Exact transferred tensors: 876
HAT-S2 parameters: 21,077,108


## Train HAT-S2

In [9]:

from copy import deepcopy
from pytorch_msssim import ssim
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.model = deepcopy(model).eval()

        for parameter in self.model.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, source_model):
        source_state = source_model.state_dict()
        ema_state = self.model.state_dict()

        for key, value in ema_state.items():
            source_value = source_state[key].detach()

            if value.dtype.is_floating_point:
                value.mul_(self.decay).add_(
                    source_value,
                    alpha=1 - self.decay,
                )
            else:
                value.copy_(source_value)


def spectral_loss(prediction, target):
    dot = torch.sum(prediction * target, dim=1)

    denominator = torch.clamp(
        torch.linalg.vector_norm(prediction, dim=1)
        * torch.linalg.vector_norm(target, dim=1),
        min=1e-8,
    )

    return (
        1.0
        - torch.clamp(
            dot / denominator,
            -1,
            1,
        )
    ).mean()


def hat_loss(prediction, target, lr):
    charbonnier = torch.sqrt(
        (prediction - target) ** 2
        + 1e-6
    ).mean()

    structural = 1.0 - ssim(
        prediction,
        target,
        data_range=1.0,
        size_average=True,
    )

    spectral = spectral_loss(
        prediction,
        target,
    )

    low_frequency = F.l1_loss(
        F.interpolate(
            prediction,
            size=lr.shape[-2:],
            mode="area",
        ),
        lr,
    )

    return (
        charbonnier
        + 0.04 * structural
        + 0.02 * spectral
        + 0.05 * low_frequency
    )


optimizer = AdamW(
    hat_s2.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=max(TARGET_EPOCHS, 1),
    eta_min=MIN_LR,
)

ema = ModelEMA(hat_s2, decay=EMA_DECAY)

start_epoch = 1
best_psnr = -float("inf")
history = []

if HAT_LAST_PATH.exists():
    checkpoint = torch.load(
        HAT_LAST_PATH,
        map_location=device,
        weights_only=False,
    )

    hat_s2.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    ema.model.load_state_dict(
        checkpoint["ema_state_dict"],
        strict=True,
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_psnr = checkpoint["best_psnr"]
    history = checkpoint.get("history", [])

    print("Resume epoch:", start_epoch)


Resume epoch: 2


In [10]:

def psnr_value(prediction, target):
    mse = F.mse_loss(prediction, target)

    return float(
        10.0
        * torch.log10(
            1.0
            / torch.clamp(mse, min=1e-12)
        ).item()
    )


@torch.inference_mode()
def validate_hat():
    ema.model.eval()

    values = {
        "psnr": [],
        "ssim": [],
        "sam": [],
    }

    for batch in tqdm(
        val_loader,
        desc="HAT validation",
        leave=False,
    ):
        lr = batch["lr"].to(device)
        hr = batch["hr"].to(device)

        prediction = ema.model(lr)

        values["psnr"].append(
            psnr_value(prediction, hr)
        )

        values["ssim"].append(
            float(
                ssim(
                    prediction,
                    hr,
                    data_range=1.0,
                    size_average=True,
                ).item()
            )
        )

        dot = torch.sum(
            prediction * hr,
            dim=1,
        )

        denominator = torch.clamp(
            torch.linalg.vector_norm(prediction, dim=1)
            * torch.linalg.vector_norm(hr, dim=1),
            min=1e-8,
        )

        sam = (
            torch.acos(
                torch.clamp(
                    dot / denominator,
                    -1 + 1e-7,
                    1 - 1e-7,
                )
            )
            * 180.0
            / math.pi
        ).mean()

        values["sam"].append(
            float(sam.item())
        )

    return {
        key: float(np.mean(current))
        for key, current in values.items()
    }


for epoch in range(start_epoch, TARGET_EPOCHS + 1):
    hat_s2.train()
    optimizer.zero_grad(set_to_none=True)

    losses = []

    progress = tqdm(
        train_loader,
        desc=f"HAT-S2 {epoch}/{TARGET_EPOCHS}",
        leave=False,
    )

    for batch_index, batch in enumerate(progress, start=1):
        lr = batch["lr"].to(device)
        hr = batch["hr"].to(device)

        prediction = hat_s2(lr)
        loss = hat_loss(prediction, hr, lr)

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite HAT loss.")

        (loss / ACCUMULATION_STEPS).backward()

        should_step = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(train_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                hat_s2.parameters(),
                max_norm=GRADIENT_CLIP,
            )

            optimizer.step()
            ema.update(hat_s2)
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())
        progress.set_postfix(loss=f"{loss.item():.5f}")

    scheduler.step()
    validation = validate_hat()

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "validation": validation,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

    print(
        f"Epoch {epoch:03d}"
        f" | Train {np.mean(losses):.6f}"
        f" | PSNR {validation['psnr']:.3f}"
        f" | SSIM {validation['ssim']:.5f}"
        f" | SAM {validation['sam']:.3f}°"
    )

    checkpoint = {
        "model_state_dict": hat_s2.state_dict(),
        "ema_state_dict": ema.model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch,
        "best_psnr": best_psnr,
        "history": history,
        "data_mode": DATA_MODE,
    }

    torch.save(
        checkpoint,
        HAT_LAST_PATH,
    )

    if validation["psnr"] > best_psnr:
        best_psnr = validation["psnr"]
        checkpoint["best_psnr"] = best_psnr

        torch.save(
            checkpoint,
            HAT_BEST_PATH,
        )

        print("Saved best HAT-S2.")

    with open(
        HAT_HISTORY_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            history,
            file,
            indent=2,
            ensure_ascii=False,
        )


HAT-S2 2/2:   0%|          | 0/2432 [00:00<?, ?it/s]

HAT validation:   0%|          | 0/302 [00:00<?, ?it/s]

Epoch 002 | Train 0.101004 | PSNR 14.000 | SSIM 0.55829 | SAM 7.403°
Saved best HAT-S2.


## تحميل OpenSR-SRGAN RGB-NIR preset

In [11]:

from opensr_srgan import load_inference_model

srgan_model = load_inference_model(
    "RGB-NIR",
    map_location="cuda",
)

if hasattr(srgan_model, "eval"):
    srgan_model.eval()

print("OpenSR-SRGAN RGB-NIR preset loaded.")


config_RGB-NIR.yaml:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

RGB-NIR_4band_inference.ckpt: reconstructing file:   0%|          |  0.00B / 47.1MB            

RGB-NIR_4band_inference.ckpt: downloading bytes:           |  0.00B            

OpenSR-SRGAN RGB-NIR preset loaded.


In [12]:

def unwrap_prediction(output):
    if isinstance(output, dict):
        for key in [
            "sr",
            "prediction",
            "output",
            "pred",
        ]:
            if key in output:
                return output[key]

        return next(iter(output.values()))

    if isinstance(output, (tuple, list)):
        return output[0]

    return output


@torch.inference_mode()
def run_srgan(lr):
    if hasattr(srgan_model, "predict_step"):
        try:
            output = srgan_model.predict_step(
                lr,
                0,
            )
        except TypeError:
            output = srgan_model.predict_step(lr)
    else:
        output = srgan_model(lr)

    prediction = unwrap_prediction(output)

    if prediction.ndim == 3:
        prediction = prediction.unsqueeze(0)

    return prediction.to(lr.device).float().clamp(0, 1)


## Quick held-out SEN2NAIP comparison

In [13]:

best_hat_checkpoint = torch.load(
    HAT_BEST_PATH,
    map_location=device,
    weights_only=False,
)

hat_s2.load_state_dict(
    best_hat_checkpoint["ema_state_dict"],
    strict=True,
)

hat_s2.eval()

quick_rows = []

models = {
    "Bicubic": lambda lr: F.interpolate(
        lr,
        scale_factor=4,
        mode="bicubic",
        align_corners=False,
    ).clamp(0, 1),
    "HAT-S2": lambda lr: hat_s2(lr),
    "OpenSR-SRGAN": run_srgan,
}

quick_values = {
    name: {
        "psnr": [],
        "ssim": [],
        "sam": [],
    }
    for name in models
}

with torch.inference_mode():
    for batch in tqdm(
        test_loader,
        desc="SEN2NAIP Test",
    ):
        lr = batch["lr"].to(device)
        hr = batch["hr"].to(device)

        for name, inference_function in models.items():
            prediction = inference_function(lr)

            if prediction.shape[-2:] != hr.shape[-2:]:
                prediction = F.interpolate(
                    prediction,
                    size=hr.shape[-2:],
                    mode="bicubic",
                    align_corners=False,
                )

            quick_values[name]["psnr"].append(
                psnr_value(prediction, hr)
            )

            quick_values[name]["ssim"].append(
                float(
                    ssim(
                        prediction,
                        hr,
                        data_range=1.0,
                        size_average=True,
                    ).item()
                )
            )

            dot = torch.sum(
                prediction * hr,
                dim=1,
            )

            denominator = torch.clamp(
                torch.linalg.vector_norm(prediction, dim=1)
                * torch.linalg.vector_norm(hr, dim=1),
                min=1e-8,
            )

            sam = (
                torch.acos(
                    torch.clamp(
                        dot / denominator,
                        -1 + 1e-7,
                        1 - 1e-7,
                    )
                )
                * 180.0
                / math.pi
            ).mean()

            quick_values[name]["sam"].append(
                float(sam.item())
            )

for name in models:
    quick_rows.append(
        {
            "Dataset": "SEN2NAIP held-out",
            "Model": name,
            "PSNR": float(np.mean(quick_values[name]["psnr"])),
            "SSIM": float(np.mean(quick_values[name]["ssim"])),
            "SAM": float(np.mean(quick_values[name]["sam"])),
        }
    )

import pandas as pd

display(
    pd.DataFrame(quick_rows).round(6)
)


SEN2NAIP Test:   0%|          | 0/117 [00:00<?, ?it/s]

,Dataset,Model,PSNR,SSIM,SAM
0,SEN2NAIP held-out,Bicubic,9.442688,0.308025,22.239005
1,SEN2NAIP held-out,HAT-S2,14.122417,0.549677,8.349316
2,SEN2NAIP held-out,OpenSR-SRGAN,9.436414,0.305998,22.260940


## Final OpenSR benchmark

In [14]:

import opensr_test

BENCHMARK_DATASETS = [
    "naip",
    "spot",
    "spain_crops",
    "spain_urban",
]

MAX_IMAGES_PER_DATASET = None
# Set a small integer for a quick smoke test.

def pad_to_window(lr, window=16):
    height, width = lr.shape[-2:]

    padded_height = (
        math.ceil(height / window)
        * window
    )

    padded_width = (
        math.ceil(width / window)
        * window
    )

    pad_bottom = padded_height - height
    pad_right = padded_width - width

    padded = F.pad(
        lr,
        (0, pad_right, 0, pad_bottom),
        mode="reflect",
    )

    return padded, height, width


@torch.inference_mode()
def run_hat_arbitrary(lr):
    padded, original_height, original_width = pad_to_window(
        lr,
        window=16,
    )

    prediction = hat_s2(padded)

    return prediction[
        :,
        :,
        :original_height * 4,
        :original_width * 4,
    ]


benchmark_rows = []

for dataset_name in BENCHMARK_DATASETS:
    dataset = opensr_test.load(
        dataset_name
    )

    lr_array = dataset["L2A"]
    hr_array = dataset["HRharm"]

    number_of_images = len(lr_array)

    if MAX_IMAGES_PER_DATASET is not None:
        number_of_images = min(
            number_of_images,
            MAX_IMAGES_PER_DATASET,
        )

    dataset_values = {
        "HAT-S2": {
            "psnr": [],
            "ssim": [],
            "sam": [],
        },
        "OpenSR-SRGAN": {
            "psnr": [],
            "ssim": [],
            "sam": [],
        },
    }

    for index in tqdm(
        range(number_of_images),
        desc=f"OpenSR {dataset_name}",
    ):
        lr = torch.from_numpy(
            np.asarray(
                lr_array[index, 0:4],
                dtype=np.float32,
            )
        ).unsqueeze(0).to(device)

        hr = torch.from_numpy(
            np.asarray(
                hr_array[index, 0:4],
                dtype=np.float32,
            )
        ).unsqueeze(0).to(device)

        if lr.max() > 2:
            lr = lr / 10000.0

        if hr.max() > 2:
            if hr.max() <= 255:
                hr = hr / 255.0
            else:
                hr = hr / 10000.0

        lr = lr.clamp(0, 1)
        hr = hr.clamp(0, 1)

        predictions = {
            "HAT-S2": run_hat_arbitrary(lr),
            "OpenSR-SRGAN": run_srgan(lr),
        }

        for model_name, prediction in predictions.items():
            if prediction.shape[-2:] != hr.shape[-2:]:
                prediction = F.interpolate(
                    prediction,
                    size=hr.shape[-2:],
                    mode="bicubic",
                    align_corners=False,
                )

            dataset_values[model_name]["psnr"].append(
                psnr_value(prediction, hr)
            )

            dataset_values[model_name]["ssim"].append(
                float(
                    ssim(
                        prediction,
                        hr,
                        data_range=1.0,
                        size_average=True,
                    ).item()
                )
            )

            dot = torch.sum(
                prediction * hr,
                dim=1,
            )

            denominator = torch.clamp(
                torch.linalg.vector_norm(prediction, dim=1)
                * torch.linalg.vector_norm(hr, dim=1),
                min=1e-8,
            )

            sam = (
                torch.acos(
                    torch.clamp(
                        dot / denominator,
                        -1 + 1e-7,
                        1 - 1e-7,
                    )
                )
                * 180.0
                / math.pi
            ).mean()

            dataset_values[model_name]["sam"].append(
                float(sam.item())
            )

    for model_name, metrics in dataset_values.items():
        benchmark_rows.append(
            {
                "Dataset": dataset_name,
                "Model": model_name,
                "Images": number_of_images,
                "PSNR": float(np.mean(metrics["psnr"])),
                "SSIM": float(np.mean(metrics["ssim"])),
                "SAM": float(np.mean(metrics["sam"])),
            }
        )

benchmark_table = pd.DataFrame(
    benchmark_rows
)

benchmark_table.to_csv(
    BENCHMARK_CSV,
    index=False,
    encoding="utf-8-sig",
)

with open(
    BENCHMARK_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "task": "Sentinel-2 RGB-NIR x4 SISR",
            "training_data": DATA_MODE,
            "metrics": benchmark_rows,
        },
        file,
        indent=2,
        ensure_ascii=False,
    )

display(
    benchmark_table.round(6)
)

print("Saved:", BENCHMARK_CSV)
print("Saved:", BENCHMARK_JSON)


OpenSR naip:   0%|          | 0/62 [00:00<?, ?it/s]

OpenSR spot:   0%|          | 0/9 [00:00<?, ?it/s]

OpenSR spain_crops:   0%|          | 0/28 [00:00<?, ?it/s]

OpenSR spain_urban:   0%|          | 0/20 [00:00<?, ?it/s]

,Dataset,Model,Images,PSNR,SSIM,SAM
0,naip,HAT-S2,62,18.900688,0.681201,26.114998
1,naip,OpenSR-SRGAN,62,20.702702,0.724517,26.839064
2,spot,HAT-S2,9,18.550194,0.697594,19.808377
3,spot,OpenSR-SRGAN,9,20.680418,0.689537,22.205311
4,spain_crops,HAT-S2,28,18.895477,0.646155,26.349029
5,spain_crops,OpenSR-SRGAN,28,19.989611,0.676868,28.020395
6,spain_urban,HAT-S2,20,18.760209,0.550283,24.030232
7,spain_urban,OpenSR-SRGAN,20,20.839804,0.575854,25.198638


Saved: /content/drive/MyDrive/Super_Resolution_28-07-2026/Sentinel2_HAT_vs_OpenSR_SRGAN/opensr_srgan_vs_hat_metrics.csv
Saved: /content/drive/MyDrive/Super_Resolution_28-07-2026/Sentinel2_HAT_vs_OpenSR_SRGAN/opensr_srgan_vs_hat_metrics.json



# طريقة التشغيل

## 1. Smoke test

```python
DATA_MODE = "cross_sensor"
RUN_PROFILE = "smoke"
MAX_IMAGES_PER_DATASET = 3
```

## 2. تدريب HAT قوي

```python
RUN_PROFILE = "long"
MAX_IMAGES_PER_DATASET = None
```

## 3. تكبير الداتا

بدل Cross-Sensor فقط:

```python
DATA_MODE = "synthetic"
NUM_SYNTHETIC_SHARDS = 4
```

ثم يمكن رفع عدد الـShards تدريجيًا حتى `18` حسب مساحة Google Drive.

## القرار العلمي

- نتائج HAT الحالية الخاصة بـWorldView-3 لا تُقارن مباشرة مع Sentinel-SRGAN.
- الجدول الناتج من هذه الـNotebook هو المقارنة الصحيحة لأن النموذجين يعملان على:
  - نفس Sentinel-2 RGB-NIR input.
  - نفس Scale ×4.
  - نفس HR reference.
  - نفس Metrics.
